In [ ]:
!pip install textattack[tensorflow]
!pip install transformers==4.41.0
!wget --no-check-certificate http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 21.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Load GloVe embeddings
glove_path = "/content/glove.6B.100d.txt"
embedding_index = {}

with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embedding_index[word] = vector



In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.regularizers import l2
import tensorflow_datasets as tfds
import numpy as np
from tensorflow.keras.models import load_model
import pickle



# Load the saved tokenizer from the pickle file
with open("tokenizer.pickle", "rb") as handle:
   tokenizer = pickle.load(handle)

# Load Yelp Polarity dataset
dataset, info = tfds.load('yelp_polarity_reviews', with_info=True, as_supervised=True)
train_data, test_data = dataset['train'], dataset['test']

# Prepare the data
vocab_size = 50000  # Limit vocabulary to top 50,000 words
max_length = 200    # Maximum review length (truncation/padding)

# Tokenizer to convert text to sequences
#tokenizer = keras.preprocessing.text.Tokenizer(num_words=vocab_size)
#tokenizer.fit_on_texts([text.numpy().decode('utf-8') for text, _ in train_data])

# Convert text data to sequences and pad them
def preprocess_dataset(dataset):
    texts, labels = [], []
    for text, label in dataset:
        texts.append(text.numpy().decode('utf-8'))
        labels.append(label.numpy())
    sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
    return padded_sequences, np.array(labels)

# Preprocess training and testing data
tr_x, tr_y = preprocess_dataset(train_data)
te_x, te_y = preprocess_dataset(test_data)



Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/yelp_polarity_reviews/incomplete.VCCI20_0.2.0/yelp_polarity_reviews-train.…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/yelp_polarity_reviews/incomplete.VCCI20_0.2.0/yelp_polarity_reviews-test.t…

Dataset yelp_polarity_reviews downloaded and prepared to /root/tensorflow_datasets/yelp_polarity_reviews/0.2.0. Subsequent calls will reuse this data.


In [ ]:
# Load the model
model = load_model('NLPModel.h5')

In [ ]:
vocab_size = 50000
# Load GloVe 6B embeddings
glove_path = "/content/glove.6B.100d.txt"  # Update with the correct path
embedding_dim = 100  # Choose dimension (50d, 100d, 200d, or 300d)

# Create a dictionary mapping words to vectors
embeddings_index = {}
with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

# Create an embedding matrix for the tokenizer’s vocabulary
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

def predict_sentiment(model, text, tokenizer, max_length=200):
    # Convert text to sequence using the tokenizer
    sequence = tokenizer.texts_to_sequences([text])

    # Pad the sequence to the fixed length
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')

    # Get prediction probability
    prediction = model.predict(padded_sequence)[0][0]

    # Classify sentiment
    sentiment = "Positive" if prediction > 0.5 else "Negative"

    # Confidence score
    confidence = prediction if sentiment == "Positive" else 1 - prediction

    return sentiment, confidence


In [ ]:
# ========================
# STABLE VIRTUAL ADVERSARIAL TRAINING DEFENSE
# ========================
print("\nImplementing Stable Virtual Adversarial Training Defense")

class VATModel(tf.keras.Model):
    def __init__(self, base_model, epsilon=2, xi=1e-6, ip=1):
        super().__init__()
        self.embedding = base_model.layers[0]
        self.encoder = tf.keras.Sequential(base_model.layers[1:])
        self.epsilon = epsilon  # Reduced perturbation
        self.xi = xi            # Smaller step size
        self.ip = ip            # Fewer power iterations

    def _normalize(self, x):
        """Numerically stable normalization"""
        norms = tf.sqrt(tf.reduce_mean(tf.square(x), axis=-1, keepdims=True))
        return x / (norms + 1e-10)

    def call(self, inputs, training=False):
        x = self.embedding(inputs)
        return self.encoder(x, training=training)

    def train_step(self, data):
        x, y = data

        # Original forward pass
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            ce_loss = self.compiled_loss(y, y_pred)
        y_pred_detached = tf.stop_gradient(y_pred)

        # Adversarial perturbation calculation
        embeddings = self.embedding(x)
        d = tf.random.normal(tf.shape(embeddings))

        # Find perturbation direction
        for _ in range(self.ip):
            d = self.xi * self._normalize(d)
            with tf.GradientTape() as tape_adv:
                tape_adv.watch(d)
                perturbed_embeds = embeddings + d
                y_hat = self.encoder(perturbed_embeds, training=True)

                y_pred_detached_clipped = tf.clip_by_value(y_pred_detached, 1e-7, 1-1e-7)
                y_hat_clipped = tf.clip_by_value(y_hat, 1e-7, 1-1e-7)
                kl_loss = tf.reduce_mean(
                    y_hat_clipped * tf.math.log(y_hat_clipped/y_pred_detached_clipped) +
                    (1 - y_hat_clipped) * tf.math.log((1 - y_hat_clipped)/(1 - y_pred_detached_clipped))
              )


            grad = tape_adv.gradient(kl_loss, d)
            d = tf.stop_gradient(self.xi * self._normalize(grad))

        # Apply final perturbation
        r_adv = self.epsilon * self._normalize(d)

        # Adversarial training step
        with tf.GradientTape() as tape_total:
            y_pred_adv = self.encoder(embeddings + r_adv, training=True)

            y_pred_detached_clipped = tf.clip_by_value(y_pred_detached, 1e-7, 1-1e-7)
            y_pred_adv_clipped = tf.clip_by_value(y_pred_adv, 1e-7, 1-1e-7)
            vat_loss = tf.reduce_mean(
                y_pred_detached_clipped * tf.math.log(y_pred_detached_clipped/y_pred_adv_clipped) +
            (1 - y_pred_detached_clipped) * tf.math.log((1 - y_pred_detached_clipped)/(1 - y_pred_adv_clipped))
        )


            # Balanced loss weighting
            total_loss = ce_loss + 1 * vat_loss  # Reduced VAT weight

        # Gradient handling
        gradients = tape_total.gradient(total_loss, self.trainable_variables)
        gradients = [
            tf.clip_by_value(g, -0.5, 0.5) if g is not None
            else tf.zeros_like(v)
            for g, v in zip(gradients, self.trainable_variables)
        ]

        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        # Update metrics
        self.compiled_metrics.update_state(y, y_pred)
        return {m.name: m.result() for m in self.metrics}


def evaluate_model(model, X, y, batch_size=512):
    """Evaluate model on clean data"""
    if len(X) == 0:
        return 0.0
    _, acc = model.evaluate(X, y, batch_size=batch_size, verbose=0)
    return acc


# Initialize and compile with learning rate schedule
vat_model = VATModel(model, epsilon=2.0, xi=1e-6, ip=1)
vat_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=1e-4,
            decay_steps=1000
        )
    ),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train with validation split
print("Training with Virtual Adversarial Defense...")
history = vat_model.fit(
    tr_x[:5000], tr_y[:5000],
    validation_split=0.2,
    epochs=8,
    batch_size=32,
)
model = load_model('NLPModel.h5')




Implementing Stable Virtual Adversarial Training Defense
Training with Virtual Adversarial Defense...
Epoch 1/8


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:667: UserWarning: `model.compiled_loss()` is deprecated. Instead, use `model.compute_loss(x, y, y_pred, sample_weight, training)`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:642: UserWarning: `model.compiled_metrics()` is deprecated. Instead, use e.g.:
```
for metric in self.metrics:
    metric.update_state(y, y_pred)
```

  return self._compiled_metrics_update_state(
/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:667: UserWarning: `model.compiled_loss()` is deprecated. Instead, use `model.compute_loss(x, y, y_pred, sample_weight, training)`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:642: UserWarning: `model.compiled_metrics()` is deprecated. Instead, use e.g.:
```
for metric in self.metrics:
    metric.update_state(y, y_pred)
```

  return self._compiled_metrics_upda

125/125 ━━━━━━━━━━━━━━━━━━━━ 73s 457ms/step - accuracy: 0.9696 - loss: 0.5089 - val_accuracy: 0.9560 - val_loss: 0.1398
Epoch 2/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 86s 489ms/step - accuracy: 0.9675 - loss: 0.5285 - val_accuracy: 0.9590 - val_loss: 0.1587
Epoch 3/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 75s 435ms/step - accuracy: 0.9660 - loss: 0.5463 - val_accuracy: 0.9580 - val_loss: 0.1592
Epoch 4/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 84s 450ms/step - accuracy: 0.9667 - loss: 0.5232 - val_accuracy: 0.9540 - val_loss: 0.1588
Epoch 5/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 77s 415ms/step - accuracy: 0.9627 - loss: 0.5322 - val_accuracy: 0.9540 - val_loss: 0.1580
Epoch 6/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 54s 434ms/step - accuracy: 0.9637 - loss: 0.5269 - val_accuracy: 0.9540 - val_loss: 0.1574
Epoch 7/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 80s 417ms/step - accuracy: 0.9688 - loss: 0.5291 - val_accuracy: 0.9540 - val_loss: 0.1572
Epoch 8/8
125/125 ━━━━━━━━━━━━━━━━━━━━ 85s 437ms/step - accuracy: 0.9624 - loss: 0.5131 - val_accura

In [ ]:
# ========================
# CHARACTER PURIFICATION DEFENSE - CORRECTED VERSION
# ========================
import re
from collections import defaultdict

class TextAttackCharacterPurifier:
    def __init__(self, tokenizer, max_edit_distance=3):
        self.tokenizer = tokenizer
        # Create vocabulary set from tokenizer
        self.vocab_set = set(tokenizer.word_index.keys())
        self.max_edit_distance = max_edit_distance

        # Pre-compile patterns for your specific TextAttack attacks
        self.repeated_char_pattern = re.compile(r'(.)\1{2,}')
        self.space_pattern = re.compile(r'\s+')

        # Build character frequency map
        self._build_char_frequency_map()

        # Character substitutions specific to your attacks
        self.char_substitutions = {
            # Homoglyph substitutions from TextBugger
            '0': 'o', '1': 'l', '5': 's', '3': 'e', '4': 'a',
            '@': 'a', '$': 's', '!': 'i', '7': 't', '8': 'b',
            # Common visual confusions
            'rn': 'm', 'vv': 'w', 'cl': 'd'
        }

        # Use the same stopwords as your attacks
        self.protected_words = {
               "a", "about", "above", "across", "after", "afterwards", "again", "against", "ain", "all", "almost", "alone", "along", "already",
            "also", "although", "am", "among", "amongst", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere",
            "are", "aren", "aren't", "around", "as", "at", "back", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides",
            "between", "beyond", "both", "but", "by", "can", "cannot", "could", "couldn", "couldn't", "d", "didn", "didn't", "doesn", "doesn't", "don",
            "don't", "down", "due", "during", "either", "else", "elsewhere", "empty", "enough", "even", "ever", "everyone", "everything", "everywhere",
            "except", "first", "for", "former", "formerly", "from", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "he", "hence", "her", "here",
            "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "if", "in", "indeed",
            "into", "is", "isn", "isn't", "it", "it's", "its", "itself", "just", "latter", "latterly", "least", "ll", "may", "me", "meanwhile", "mightn",
            "mightn't", "mine", "more", "moreover", "most", "mostly", "must", "mustn", "mustn't", "my", "myself", "namely", "needn", "needn't", "neither",
            "never", "nevertheless", "next", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "o", "of", "off", "on", "once",
            "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "per", "please", "s", "same", "shan",
            "shan't", "she", "she's", "should've", "shouldn", "shouldn't", "somehow", "something", "sometime", "somewhere", "such", "t", "than", "that",
            "that'll", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon",
            "these", "they", "this", "those", "through", "throughout", "thru", "thus", "to", "too", "toward", "towards", "under", "unless", "until", "up",
            "upon", "used", "ve", "was", "wasn", "wasn't", "we", "were", "weren", "weren't", "what", "whatever", "when", "whence", "whenever", "where",
            "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole",
            "whom", "whose", "why", "with", "within", "without", "won", "won't", "would", "wouldn", "wouldn't", "y", "yet", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"}

    def _build_char_frequency_map(self):
        """Build character frequency for intelligent corrections using most frequent words"""
        self.char_freq = defaultdict(int)

        # Use most frequent words from tokenizer if available
        if hasattr(self.tokenizer, 'word_index'):
            vocab_sample = list(self.tokenizer.word_index.keys())[:5000]
        else:
            # Fallback to existing vocab_set
            vocab_sample = list(self.vocab_set)[:5000] if len(self.vocab_set) > 5000 else list(self.vocab_set)

        for word in vocab_sample:
            for char in word.lower():
                if char.isalpha():
                    self.char_freq[char] += 1

    # Sort by frequency
        self.common_chars = sorted(self.char_freq.keys(),
                               key=lambda x: self.char_freq[x], reverse=True)[:26]

    def purify_text_for_textattack(self, text):
        """Main purification function for your TextAttack setup"""
        # Handle space insertions first (TextBugger attack)
        text = self.space_pattern.sub(' ', text)

        words = text.split()
        purified_words = []

        for word in words:
            # Skip protected stopwords
            if word.lower() in self.protected_words:
                purified_words.append(word)
                continue

            # Quick vocabulary check
            if word.lower() in self.vocab_set:
                purified_words.append(word)
                continue

            # Apply purification
            corrected = self._purify_word(word)
            purified_words.append(corrected)

        return " ".join(purified_words)

    def _purify_word(self, word):
        """Purify a single word against your specific attacks"""
        original = word

        # Step 1: Fix homoglyph substitutions (TextBugger)
        for fake_char, real_char in self.char_substitutions.items():
            word = word.replace(fake_char, real_char)

        # Step 2: Remove excessive repetitions
        word = self.repeated_char_pattern.sub(r'\1', word)

        # Step 3: Check if fixed
        if word.lower() in self.vocab_set:
            return word

        # Step 4: Try single character corrections (DeepWordBug)
        corrected = self._single_edit_correction(word)
        if corrected:
            return corrected

        # Step 5: Try adjacent swaps (DeepWordBug)
        swapped = self._fix_adjacent_swaps(word)
        if swapped:
            return swapped

        # Return original if no correction found
        return original

    def _single_edit_correction(self, word):
        """Handle single character operations from DeepWordBug"""
        # Try deletion (for insertion attacks)
        for i in range(len(word)):
            candidate = word[:i] + word[i+1:]
            if len(candidate) > 1 and candidate.lower() in self.vocab_set:
                return candidate

        # Try substitution (for substitution attacks)
        for i in range(len(word)):
            for char in self.common_chars[:26]:  # Limit for speed
                candidate = word[:i] + char + word[i+1:]
                if candidate.lower() in self.vocab_set:
                    return candidate

        # Try insertion (for deletion attacks)
        for i in range(len(word) + 1):
            for char in self.common_chars[:26]:  # Most common chars only
                candidate = word[:i] + char + word[i:]
                if len(candidate) <= len(word) + 2 and candidate.lower() in self.vocab_set:
                    return candidate

        return None

    def _fix_adjacent_swaps(self, word):
        """Fix adjacent character swaps from DeepWordBug"""
        for i in range(len(word) - 1):
            chars = list(word)
            chars[i], chars[i+1] = chars[i+1], chars[i]
            candidate = ''.join(chars)
            if candidate.lower() in self.vocab_set:
                return candidate
        return None

# CORRECTED EVALUATION FUNCTION
def evaluate_adversarial_with_counts(defense_model, attack_results, tokenizer, max_length, use_purification=False):
    """
    Evaluate defense with optional purification
    """
    # Initialize purifier if requested
    purifier = TextAttackCharacterPurifier(tokenizer) if use_purification else None

    perturbed_texts = []
    true_labels = []

    # Collect adversarial examples
    for res in attack_results:
        if hasattr(res, "perturbed_result") and hasattr(res.perturbed_result, "attacked_text"):
            text = res.perturbed_result.attacked_text.text
            # Apply purification if enabled
            if purifier:
                text = purifier.purify_text_for_textattack(text)
            perturbed_texts.append(text)
            true_labels.append(res.original_result.ground_truth_output)
        elif hasattr(res, "original_result") and hasattr(res.original_result, "attacked_text"):
            text = res.original_result.attacked_text.text
            if purifier:
                text = purifier.purify_text_for_textattack(text)
            perturbed_texts.append(text)
            true_labels.append(res.original_result.ground_truth_output)

    if not perturbed_texts:
        return {
            "accuracy": 0.0,
            "defense_successful": 0,
            "defense_failed": 0,
            "skipped": 0,
            "total_attempted": 0
        }

    # Convert to model inputs
    sequences = tokenizer.texts_to_sequences(perturbed_texts)
    adv_x = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

    # Get defense model predictions
    defense_preds = defense_model.predict(adv_x, verbose=0)
    predicted_labels = (defense_preds > 0.5).flatten().astype(int)
    true_labels_array = np.array(true_labels)

    # Calculate metrics
    defense_adv_acc = np.mean(predicted_labels == true_labels_array)
    correct_predictions = (predicted_labels == true_labels_array)
    defense_successful = np.sum(correct_predictions)
    defense_failed = len(true_labels) - defense_successful

    # Count skipped from original results
    skipped_attacks = sum(1 for res in attack_results if hasattr(res, 'status') and res.status == 'skipped')

    return {
        "accuracy": defense_adv_acc,
        "defense_successful": defense_successful,
        "defense_failed": defense_failed,
        "skipped": skipped_attacks,
        "total_attempted": len(true_labels)
    }


In [ ]:
import numpy as np
import tensorflow as tf
import textattack
from textattack import Attack, Attacker, AttackArgs
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
    InputColumnModification
)
from textattack.constraints.grammaticality import PartOfSpeech
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder

from textattack.transformations import WordSwapEmbedding
from textattack.search_methods import GreedyWordSwapWIR
from textattack.goal_functions import UntargetedClassification
from textattack.models.wrappers import ModelWrapper
from textattack.shared import WordEmbedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from textattack.models.wrappers import ModelWrapper
import numpy as np
class CustomKerasModelWrapper(ModelWrapper):
    def __init__(self, model, tokenizer, max_length=200):
        self.model = model
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, text_input_list):
        sequences = self.tokenizer.texts_to_sequences(text_input_list)
        padded = pad_sequences(sequences, maxlen=self.max_length, padding='post', truncating='post')

        # Get raw probabilities (shape [batch_size, 1])
        preds = self.model.predict(padded)

        # Convert to logits for both classes (shape [batch_size, 2])
        # For binary classification: class 0 = 1 - prob, class 1 = prob
        return np.concatenate([1 - preds, preds], axis=1)


model_wrapper = CustomKerasModelWrapper(model, tokenizer, max_length=max_length)
import numpy as np

# Take first 100 preprocessed samples
sample_size = 1000
te_x_sample = te_x[:sample_size]
te_y_sample = te_y[:sample_size]

# Reconstruct raw texts from tokenizer (approximate)
reverse_word_map = dict(map(reversed, tokenizer.word_index.items()))
test_texts = [
    " ".join([reverse_word_map.get(idx, "") for idx in seq if idx != 0])
    for seq in te_x_sample
]
# Original predictions
baseline_preds = model.predict(te_x_sample)
baseline_accuracy = np.mean(np.round(baseline_preds).flatten() == te_y_sample)
print(f"Baseline Accuracy: {baseline_accuracy:.2%}")



32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step
Baseline Accuracy: 95.70%


In [ ]:
from textattack.datasets import Dataset
import nltk
from textattack.attack_results import (
    AttackResult,
    SuccessfulAttackResult,
    FailedAttackResult,
    SkippedAttackResult
)
from textattack import Attacker, AttackArgs
nltk.download('averaged_perceptron_tagger_eng')
stopwords = set(
            ["a", "about", "above", "across", "after", "afterwards", "again", "against", "ain", "all", "almost", "alone", "along", "already", "also", "although", "am", "among", "amongst", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", "are", "aren", "aren't", "around", "as", "at", "back", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", "beyond", "both", "but", "by", "can", "cannot", "could", "couldn", "couldn't", "d", "didn", "didn't", "doesn", "doesn't", "don", "don't", "down", "due", "during", "either", "else", "elsewhere", "empty", "enough", "even", "ever", "everyone", "everything", "everywhere", "except", "first", "for", "former", "formerly", "from", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "he", "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "if", "in", "indeed", "into", "is", "isn", "isn't", "it", "it's", "its", "itself", "just", "latter", "latterly", "least", "ll", "may", "me", "meanwhile", "mightn", "mightn't", "mine", "more", "moreover", "most", "mostly", "must", "mustn", "mustn't", "my", "myself", "namely", "needn", "needn't", "neither", "never", "nevertheless", "next", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "o", "of", "off", "on", "once", "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "per", "please", "s", "same", "shan", "shan't", "she", "she's", "should've", "shouldn", "shouldn't", "somehow", "something", "sometime", "somewhere", "such", "t", "than", "that", "that'll", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon", "these", "they", "this", "those", "through", "throughout", "thru", "thus", "to", "too", "toward", "towards", "under", "unless", "until", "up", "upon", "used", "ve", "was", "wasn", "wasn't", "we", "were", "weren", "weren't", "what", "whatever", "when", "whence", "whenever", "where", "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole", "whom", "whose", "why", "with", "within", "without", "won", "won't", "would", "wouldn", "wouldn't", "y", "yet", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"]
        )
use_constraint = UniversalSentenceEncoder(
            threshold=0.840845057,
            metric="angular",
            compare_against_original=True,
            window_size=15,
            skip_text_shorter_than_window=True,
        )
constraints_TextFooler = [
    RepeatModification(),
    StopwordModification(stopwords=stopwords),
    WordEmbeddingDistance(min_cos_sim=0.5),
    PartOfSpeech(allow_verb_noun_swap=False),
    use_constraint
]
attack_TextFooler = Attack(
    UntargetedClassification(model_wrapper),
    constraints_TextFooler,
    WordSwapEmbedding(max_candidates=50),
    GreedyWordSwapWIR(wir_method="delete")
)
# Filter to samples the model originally got RIGHT
original_correct_mask = (np.round(baseline_preds).flatten() == te_y_sample)
originally_correct_indices = np.where(original_correct_mask)[0]
num_originally_correct = len(originally_correct_indices)

print(f"Samples originally correct: {num_originally_correct}/{sample_size}")

# Subset to only these correctly-classified samples
attack_texts = [test_texts[i] for i in originally_correct_indices]
attack_labels = te_y_sample[originally_correct_indices]
attack_labels_fixed = [int(label) for label in attack_labels]  # Convert to Python int

attack_args = AttackArgs(
    num_examples=num_originally_correct,
    log_to_csv="attack_log_TextFooler.csv",
    disable_stdout=True
)
attacker_TextFooler = Attacker(attack_TextFooler, Dataset(list(zip(attack_texts, attack_labels_fixed))), attack_args)  # Use fixed labels
attack_results_TextFooler = attacker_TextFooler.attack_dataset()


# Calculate metrics per paper
successful_attacks = sum(1 for res in attack_results_TextFooler if isinstance(res, SuccessfulAttackResult))
skipped_attacks = sum(1 for res in attack_results_TextFooler if isinstance(res, SkippedAttackResult))
failed_attacks = num_originally_correct - successful_attacks - skipped_attacks

# Paper-style metrics
original_accuracy = num_originally_correct / sample_size  # Baseline accuracy
attack_success_rate = successful_attacks / (num_originally_correct - skipped_attacks)  # ASR
accuracy_under_attack = (num_originally_correct - successful_attacks) / sample_size  # AuA

print(f"""
+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | {original_accuracy:.2%}  |
| Accuracy Under Attack     | {accuracy_under_attack:.2%}  |
| Attack Success Rate (ASR) | {attack_success_rate:.2%}  |
| Successful Attacks        | {successful_attacks}       |
| Failed Attacks            | {failed_attacks}       |
| Skipped Attacks           | {skipped_attacks}       |
+---------------------------+-----------+
""")



[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
textattack: Unknown if model of class <class 'keras.src.models.sequential.Sequential'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
textattack: Logging to CSV at path attack_log_TextFooler.csv


Samples originally correct: 957/1000
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  False
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  True
      )
    (3): RepeatModification
    (4): StopwordModification
  (is_black_box):  True
) 




  0%|          | 0/957 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step



  0%|          | 1/957 [00:18<4:48:12, 18.09s/it]
[Succeeded / Failed / Skipped / Total] 0 / 1 / 0 / 1:   0%|          | 1/957 [00:18<4:48:36, 18.11s/it]

...


[Succeeded / Failed / Skipped / Total] 917 / 38 / 0 / 955: 100%|█████████▉| 956/957 [49:21<00:03,  3.10s/it]
[Succeeded / Failed / Skipped / Total] 918 / 38 / 0 / 956: 100%|█████████▉| 956/957 [49:21<00:03,  3.10s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step



[Succeeded / Failed / Skipped / Total] 918 / 38 / 0 / 956: 100%|██████████| 957/957 [49:25<00:00,  3.10s/it]
[Succeeded / Failed / Skipped / Total] 919 / 38 / 0 / 957: 100%|██████████| 957/957 [49:25<00:00,  3.10s/it]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 919    |
| Number of failed attacks:     | 38     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 3.97%  |
| Attack success rate:          | 96.03% |
| Average perturbed word %:     | 5.74%  |
| Average num. words per input: | 108.54 |
| Avg num queries:              | 161.92 |
+-------------------------------+--------+



+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | 95.70%  |
| Accuracy Under Attack     | 3.80%  |
| Attack Success Rate (ASR) | 96.03%  |
| Successful Attacks        | 919       |
| Failed Attacks            | 38       |
| Skipped Attacks           | 0       |
+---------------------------+-----------+



In [ ]:
print("\n=== Adversarial Evaluation TextFooler===")

# Evaluate VAT only
defense_metrics = evaluate_adversarial_with_counts(vat_model, attack_results_TextFooler, tokenizer, max_length, use_purification=False)
orig_metrics = evaluate_adversarial_with_counts(model, attack_results_TextFooler, tokenizer, max_length, use_purification=False)

# Evaluate VAT + Purification
purified_metrics = evaluate_adversarial_with_counts(vat_model, attack_results_TextFooler, tokenizer, max_length, use_purification=True)

# Calculate ASR metrics
asr_original = 1 - orig_metrics["accuracy"]
asr_defended = 1 - defense_metrics["accuracy"]
asr_purified = 1 - purified_metrics["accuracy"]
asr_reduction_vat = asr_original - asr_defended
asr_reduction_purified = asr_original - asr_purified

print(f"""
+---------------------------+-----------+-----------+-----------+
| Metric                    | Original  | VAT Only  | VAT+Purif |
+---------------------------+-----------+-----------+-----------+
| Accuracy                  | {orig_metrics["accuracy"]:.2%}    | {defense_metrics["accuracy"]:.2%}    | {purified_metrics["accuracy"]:.2%}    |
| ASR                       | {asr_original:.2%}    | {asr_defended:.2%}    | {asr_purified:.2%}    |
| ASR Reduction             | -         | {asr_reduction_vat:.2%}    | {asr_reduction_purified:.2%}    |
| Successful Defenses       | -         | {defense_metrics["defense_successful"]}       | {purified_metrics["defense_successful"]}       |
| Failed Defenses           | -         | {defense_metrics["defense_failed"]}       | {purified_metrics["defense_failed"]}       |
| Total Attempted           | {orig_metrics["total_attempted"]}      | {defense_metrics["total_attempted"]}      | {purified_metrics["total_attempted"]}      |
+---------------------------+-----------+-----------+-----------+
""")



=== Adversarial Evaluation ===

+---------------------------+-----------+-----------+-----------+
| Metric                    | Original  | VAT Only  | VAT+Purif |
+---------------------------+-----------+-----------+-----------+
| Accuracy                  | 3.97%    | 49.74%    | 49.74%    |
| ASR                       | 96.03%    | 50.26%    | 50.26%    |
| ASR Reduction             | -         | 45.77%    | 45.77%    |
| Successful Defenses       | -         | 476       | 476       |
| Failed Defenses           | -         | 481       | 481       |
| Total Attempted           | 957      | 957      | 957      |
+---------------------------+-----------+-----------+-----------+



In [ ]:
# Load the model
model = load_model('NLPModel.h5')

In [ ]:
import numpy as np
import tensorflow as tf
from textattack import Attack
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import (
    CompositeTransformation,
    WordSwapEmbedding,
    WordSwapHomoglyphSwap,
    WordSwapNeighboringCharacterSwap,
    WordSwapRandomCharacterDeletion,
    WordSwapRandomCharacterInsertion,
)
stopwords = set(
            ["a", "about", "above", "across", "after", "afterwards", "again", "against", "ain", "all", "almost", "alone", "along", "already", "also", "although", "am", "among", "amongst", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", "are", "aren", "aren't", "around", "as", "at", "back", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", "beyond", "both", "but", "by", "can", "cannot", "could", "couldn", "couldn't", "d", "didn", "didn't", "doesn", "doesn't", "don", "don't", "down", "due", "during", "either", "else", "elsewhere", "empty", "enough", "even", "ever", "everyone", "everything", "everywhere", "except", "first", "for", "former", "formerly", "from", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "he", "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "if", "in", "indeed", "into", "is", "isn", "isn't", "it", "it's", "its", "itself", "just", "latter", "latterly", "least", "ll", "may", "me", "meanwhile", "mightn", "mightn't", "mine", "more", "moreover", "most", "mostly", "must", "mustn", "mustn't", "my", "myself", "namely", "needn", "needn't", "neither", "never", "nevertheless", "next", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "o", "of", "off", "on", "once", "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "per", "please", "s", "same", "shan", "shan't", "she", "she's", "should've", "shouldn", "shouldn't", "somehow", "something", "sometime", "somewhere", "such", "t", "than", "that", "that'll", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon", "these", "they", "this", "those", "through", "throughout", "thru", "thus", "to", "too", "toward", "towards", "under", "unless", "until", "up", "upon", "used", "ve", "was", "wasn", "wasn't", "we", "were", "weren", "weren't", "what", "whatever", "when", "whence", "whenever", "where", "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole", "whom", "whose", "why", "with", "within", "without", "won", "won't", "would", "wouldn", "wouldn't", "y", "yet", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"]
        )
transformation = CompositeTransformation(
            [
                # (1) Insert: Insert a space into the word.
                # Generally, words are segmented by spaces in English. Therefore,
                # we can deceive classifiers by inserting spaces into words.
                WordSwapRandomCharacterInsertion(
                    random_one=True,
                    letters_to_insert=" ",
                    skip_first_char=True,
                    skip_last_char=True,
                ),
                # (2) Delete: Delete a random character of the word except for the first
                # and the last character.
                WordSwapRandomCharacterDeletion(
                    random_one=True, skip_first_char=True, skip_last_char=True
                ),
                # (3) Swap: Swap random two adjacent letters in the word but do not
                # alter the first or last letter. This is a common occurrence when
                # typing quickly and is easy to implement.
                WordSwapNeighboringCharacterSwap(
                    random_one=True, skip_first_char=True, skip_last_char=True
                ),
                # (4) Substitute-C (Sub-C): Replace characters with visually similar
                # characters (e.g., replacing “o” with “0”, “l” with “1”, “a” with “@”)
                # or adjacent characters in the keyboard (e.g., replacing “m” with “n”).
                WordSwapHomoglyphSwap(),
                # (5) Substitute-W
                # (Sub-W): Replace a word with its topk nearest neighbors in a
                # context-aware word vector space. Specifically, we use the pre-trained
                # GloVe model [30] provided by Stanford for word embedding and set
                # topk = 5 in the experiment.
                WordSwapEmbedding(max_candidates=5),
            ]
        )

constraints = [
    RepeatModification(),
    StopwordModification(stopwords=stopwords),
    UniversalSentenceEncoder(threshold=0.8)
]

attack_TextBugger = Attack(
    UntargetedClassification(model_wrapper),
    constraints,
    transformation,
    GreedyWordSwapWIR(wir_method="delete")
)
baseline_preds = model.predict(te_x_sample)
baseline_accuracy = np.mean(np.round(baseline_preds).flatten() == te_y_sample)
print(f"Baseline Accuracy: {baseline_accuracy:.2%}")



textattack: Unknown if model of class <class 'keras.src.models.sequential.Sequential'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step
Baseline Accuracy: 95.70%


In [ ]:
from textattack.datasets import Dataset
import nltk
from textattack.attack_results import (
    AttackResult,
    SuccessfulAttackResult,
    FailedAttackResult,
    SkippedAttackResult
)
nltk.download('averaged_perceptron_tagger_eng')
# Filter to samples the model originally got RIGHT
original_correct_mask = (np.round(baseline_preds).flatten() == te_y_sample)
originally_correct_indices = np.where(original_correct_mask)[0]
num_originally_correct = len(originally_correct_indices)

print(f"Samples originally correct: {num_originally_correct}/{sample_size}")

 #Subset to only these correctly-classified samples
attack_texts = [test_texts[i] for i in originally_correct_indices]
attack_labels = te_y_sample[originally_correct_indices]
attack_labels_fixed = [int(label) for label in attack_labels]  # Convert to Python int

# Attack only these samples
from textattack import Attacker, AttackArgs

attack_args_TextBugger = AttackArgs(
    num_examples=num_originally_correct,
    log_to_csv="attack_log_TextBugger.csv",
    disable_stdout=True
)
attacker_TextBugger = Attacker(attack_TextBugger, Dataset(list(zip(attack_texts, attack_labels_fixed))), attack_args_TextBugger)  # Use fixed labels
attack_results_TextBugger = attacker_TextBugger.attack_dataset()


# Calculate metrics per paper
successful_attacks = sum(1 for res in attack_results_TextBugger if isinstance(res, SuccessfulAttackResult))
skipped_attacks = sum(1 for res in attack_results_TextBugger if isinstance(res, SkippedAttackResult))
failed_attacks = num_originally_correct - successful_attacks - skipped_attacks

# Paper-style metrics
original_accuracy = num_originally_correct / sample_size  # Baseline accuracy
attack_success_rate = successful_attacks / (num_originally_correct - skipped_attacks)  # ASR
accuracy_under_attack = (num_originally_correct - successful_attacks) / sample_size  # AuA

print(f"""
+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | {original_accuracy:.2%}  |
| Accuracy Under Attack     | {accuracy_under_attack:.2%}  |
| Attack Success Rate (ASR) | {attack_success_rate:.2%}  |
| Successful Attacks        | {successful_attacks}       |
| Failed Attacks            | {failed_attacks}       |
| Skipped Attacks           | {skipped_attacks}       |
+---------------------------+-----------+
""")


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
textattack: Logging to CSV at path attack_log_TextBugger.csv


Samples originally correct: 957/1000
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (2): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (3): WordSwapHomoglyphSwap
    (4): WordSwapEmbedding(
        (max_candidates):  5
        (embedding):  WordEmbedding
      )
    )
  (constraints): 
    (0): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.8
        (window_size):  inf
        (skip_text_shorter_than_window):  False
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 




  0%|          | 0/957 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step



  0%|          | 1/957 [00:15<4:08:13, 15.58s/it]
[Succeeded / Failed / Skipped / Total] 0 / 1 / 0 / 1:   0%|          | 1/957 [00:15<4:08:41, 15.61s/it]

...


[Succeeded / Failed / Skipped / Total] 885 / 70 / 0 / 955: 100%|█████████▉| 956/957 [44:12<00:02,  2.77s/it]
[Succeeded / Failed / Skipped / Total] 886 / 70 / 0 / 956: 100%|█████████▉| 956/957 [44:12<00:02,  2.77s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step



[Succeeded / Failed / Skipped / Total] 886 / 70 / 0 / 956: 100%|██████████| 957/957 [44:15<00:00,  2.77s/it]
[Succeeded / Failed / Skipped / Total] 887 / 70 / 0 / 957: 100%|██████████| 957/957 [44:15<00:00,  2.77s/it]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 887    |
| Number of failed attacks:     | 70     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 7.31%  |
| Attack success rate:          | 92.69% |
| Average perturbed word %:     | 41.21% |
| Average num. words per input: | 108.54 |
| Avg num queries:              | 150.12 |
+-------------------------------+--------+



+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | 95.70%  |
| Accuracy Under Attack     | 7.00%  |
| Attack Success Rate (ASR) | 92.69%  |
| Successful Attacks        | 887       |
| Failed Attacks            | 70       |
| Skipped Attacks           | 0       |
+---------------------------+-----------+



In [ ]:
print("\n=== Adversarial Evaluation Text Bugger===")

# Evaluate VAT only
defense_metrics = evaluate_adversarial_with_counts(vat_model, attack_results_TextBugger, tokenizer, max_length, use_purification=False)
orig_metrics = evaluate_adversarial_with_counts(model, attack_results_TextBugger, tokenizer, max_length, use_purification=False)

# Evaluate VAT + Purification
purified_metrics = evaluate_adversarial_with_counts(vat_model, attack_results_TextBugger, tokenizer, max_length, use_purification=True)

# Calculate ASR metrics
asr_original = 1 - orig_metrics["accuracy"]
asr_defended = 1 - defense_metrics["accuracy"]
asr_purified = 1 - purified_metrics["accuracy"]
asr_reduction_vat = asr_original - asr_defended
asr_reduction_purified = asr_original - asr_purified

print(f"""
+---------------------------+-----------+-----------+-----------+
| Metric                    | Original  | VAT Only  | VAT+Purif |
+---------------------------+-----------+-----------+-----------+
| Accuracy                  | {orig_metrics["accuracy"]:.2%}    | {defense_metrics["accuracy"]:.2%}    | {purified_metrics["accuracy"]:.2%}    |
| ASR                       | {asr_original:.2%}    | {asr_defended:.2%}    | {asr_purified:.2%}    |
| ASR Reduction             | -         | {asr_reduction_vat:.2%}    | {asr_reduction_purified:.2%}    |
| Successful Defenses       | -         | {defense_metrics["defense_successful"]}       | {purified_metrics["defense_successful"]}       |
| Failed Defenses           | -         | {defense_metrics["defense_failed"]}       | {purified_metrics["defense_failed"]}       |
| Total Attempted           | {orig_metrics["total_attempted"]}      | {defense_metrics["total_attempted"]}      | {purified_metrics["total_attempted"]}      |
+---------------------------+-----------+-----------+-----------+
""")



=== Adversarial Evaluation ===

+---------------------------+-----------+-----------+-----------+
| Metric                    | Original  | VAT Only  | VAT+Purif |
+---------------------------+-----------+-----------+-----------+
| Accuracy                  | 7.31%    | 52.98%    | 57.68%    |
| ASR                       | 92.69%    | 47.02%    | 42.32%    |
| ASR Reduction             | -         | 45.66%    | 50.37%    |
| Successful Defenses       | -         | 507       | 552       |
| Failed Defenses           | -         | 450       | 405       |
| Total Attempted           | 957      | 957      | 957      |
+---------------------------+-----------+-----------+-----------+



In [ ]:
# Load the model
model = load_model('NLPModel.h5')

In [ ]:
from textattack import Attack
from textattack.constraints.overlap import LevenshteinEditDistance
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
)
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import (
    CompositeTransformation,
    WordSwapNeighboringCharacterSwap,
    WordSwapRandomCharacterDeletion,
    WordSwapRandomCharacterInsertion,
    WordSwapRandomCharacterSubstitution,
)
stopwords = set(
            ["a", "about", "above", "across", "after", "afterwards", "again", "against", "ain", "all", "almost", "alone", "along", "already", "also", "although", "am", "among", "amongst", "an", "and", "another", "any", "anyhow", "anyone", "anything", "anyway", "anywhere", "are", "aren", "aren't", "around", "as", "at", "back", "been", "before", "beforehand", "behind", "being", "below", "beside", "besides", "between", "beyond", "both", "but", "by", "can", "cannot", "could", "couldn", "couldn't", "d", "didn", "didn't", "doesn", "doesn't", "don", "don't", "down", "due", "during", "either", "else", "elsewhere", "empty", "enough", "even", "ever", "everyone", "everything", "everywhere", "except", "first", "for", "former", "formerly", "from", "hadn", "hadn't", "hasn", "hasn't", "haven", "haven't", "he", "hence", "her", "here", "hereafter", "hereby", "herein", "hereupon", "hers", "herself", "him", "himself", "his", "how", "however", "hundred", "i", "if", "in", "indeed", "into", "is", "isn", "isn't", "it", "it's", "its", "itself", "just", "latter", "latterly", "least", "ll", "may", "me", "meanwhile", "mightn", "mightn't", "mine", "more", "moreover", "most", "mostly", "must", "mustn", "mustn't", "my", "myself", "namely", "needn", "needn't", "neither", "never", "nevertheless", "next", "no", "nobody", "none", "noone", "nor", "not", "nothing", "now", "nowhere", "o", "of", "off", "on", "once", "one", "only", "onto", "or", "other", "others", "otherwise", "our", "ours", "ourselves", "out", "over", "per", "please", "s", "same", "shan", "shan't", "she", "she's", "should've", "shouldn", "shouldn't", "somehow", "something", "sometime", "somewhere", "such", "t", "than", "that", "that'll", "the", "their", "theirs", "them", "themselves", "then", "thence", "there", "thereafter", "thereby", "therefore", "therein", "thereupon", "these", "they", "this", "those", "through", "throughout", "thru", "thus", "to", "too", "toward", "towards", "under", "unless", "until", "up", "upon", "used", "ve", "was", "wasn", "wasn't", "we", "were", "weren", "weren't", "what", "whatever", "when", "whence", "whenever", "where", "whereafter", "whereas", "whereby", "wherein", "whereupon", "wherever", "whether", "which", "while", "whither", "who", "whoever", "whole", "whom", "whose", "why", "with", "within", "without", "won", "won't", "would", "wouldn", "wouldn't", "y", "yet", "you", "you'd", "you'll", "you're", "you've", "your", "yours", "yourself", "yourselves"]
        )
transformation = CompositeTransformation(
                [
                    # (1) Swap: Swap two adjacent letters in the word.
                    WordSwapNeighboringCharacterSwap(),
                    # (2) Substitution: Substitute a letter in the word with a random letter.
                    WordSwapRandomCharacterSubstitution(),
                    # (3) Deletion: Delete a random letter from the word.
                    WordSwapRandomCharacterDeletion(),
                    # (4) Insertion: Insert a random letter in the word.
                    WordSwapRandomCharacterInsertion(),
                ]
            )
constraints = [
    RepeatModification(),
    StopwordModification(stopwords=stopwords),
    LevenshteinEditDistance(30)
]

attack_DeepWordBug = Attack(
    UntargetedClassification(model_wrapper),
    constraints,
    transformation,
    GreedyWordSwapWIR()
)
# Original predictions
baseline_preds = model.predict(te_x_sample)
baseline_accuracy = np.mean(np.round(baseline_preds).flatten() == te_y_sample)
print(f"Baseline Accuracy: {baseline_accuracy:.2%}")


textattack: Unknown if model of class <class 'keras.src.models.sequential.Sequential'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step
Baseline Accuracy: 95.70%


In [ ]:
from textattack.datasets import Dataset
import nltk
from textattack.attack_results import (
    AttackResult,
    SuccessfulAttackResult,
    FailedAttackResult,
    SkippedAttackResult
)
nltk.download('averaged_perceptron_tagger_eng')
# Filter to samples the model originally got RIGHT
original_correct_mask = (np.round(baseline_preds).flatten() == te_y_sample)
originally_correct_indices = np.where(original_correct_mask)[0]
num_originally_correct = len(originally_correct_indices)

print(f"Samples originally correct: {num_originally_correct}/{sample_size}")

# Subset to only these correctly-classified samples
attack_texts = [test_texts[i] for i in originally_correct_indices]
attack_labels = te_y_sample[originally_correct_indices]
attack_labels_fixed = [int(label) for label in attack_labels]  # Convert to Python int

# Attack only these samples
from textattack import Attacker, AttackArgs

attack_args_DeepWordBug = AttackArgs(
    num_examples=num_originally_correct,
    log_to_csv="attack_log.csv",
    disable_stdout=True
)
attacker_DeepWordBug = Attacker(attack_DeepWordBug, Dataset(list(zip(attack_texts, attack_labels_fixed))), attack_args_DeepWordBug)  # Use fixed labels
attack_results_DeepWordBug = attacker_DeepWordBug.attack_dataset()


# Calculate metrics per paper
successful_attacks = sum(1 for res in attack_results_DeepWordBug if isinstance(res, SuccessfulAttackResult))
skipped_attacks = sum(1 for res in attack_results_DeepWordBug if isinstance(res, SkippedAttackResult))
failed_attacks = num_originally_correct - successful_attacks - skipped_attacks

# Paper-style metrics
original_accuracy = num_originally_correct / sample_size  # Baseline accuracy
attack_success_rate = successful_attacks / (num_originally_correct - skipped_attacks)  # ASR
accuracy_under_attack = (num_originally_correct - successful_attacks) / sample_size  # AuA

print(f"""
+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | {original_accuracy:.2%}  |
| Accuracy Under Attack     | {accuracy_under_attack:.2%}  |
| Attack Success Rate (ASR) | {attack_success_rate:.2%}  |
| Successful Attacks        | {successful_attacks}       |
| Failed Attacks            | {failed_attacks}       |
| Skipped Attacks           | {skipped_attacks}       |
+---------------------------+-----------+
""")


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
textattack: Logging to CSV at path attack_log.csv


Samples originally correct: 957/1000
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterSubstitution(
        (random_one):  True
      )
    (2): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (3): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    )
  (constraints): 
    (0): LevenshteinEditDistance(
        (max_edit_distance):  30
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 




  0%|          | 0/957 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step



  0%|          | 1/957 [00:00<11:12,  1.42it/s]
[Succeeded / Failed / Skipped / Total] 0 / 1 / 0 / 1:   0%|          | 1/957 [00:00<11:24,  1.40it/s]

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step



[Succeeded / Failed / Skipped / Total] 884 / 71 / 0 / 955: 100%|█████████▉| 956/957 [45:50<00:02,  2.88s/it]
[Succeeded / Failed / Skipped / Total] 885 / 71 / 0 / 956: 100%|█████████▉| 956/957 [45:50<00:02,  2.88s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step



[Succeeded / Failed / Skipped / Total] 885 / 71 / 0 / 956: 100%|██████████| 957/957 [45:53<00:00,  2.88s/it]
[Succeeded / Failed / Skipped / Total] 886 / 71 / 0 / 957: 100%|██████████| 957/957 [45:53<00:00,  2.88s/it]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 886    |
| Number of failed attacks:     | 71     |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 7.42%  |
| Attack success rate:          | 92.58% |
| Average perturbed word %:     | 10.43% |
| Average num. words per input: | 108.54 |
| Avg num queries:              | 95.11  |
+-------------------------------+--------+

+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | 95.70%  |
| Accuracy Under Attack     | 7.10%  |
| Attack Success Rate (ASR) | 92.58%  |
| Successful Attacks        | 886       |
| Failed Attacks            | 71       |
| Skipped Attacks           | 0       |
+---------------------------+-----------+



In [ ]:
print("\n=== Adversarial Evaluation DeepWord Bug===")

# Evaluate VAT only
defense_metrics = evaluate_adversarial_with_counts(vat_model, attack_results_DeepWordBug, tokenizer, max_length, use_purification=False)
orig_metrics = evaluate_adversarial_with_counts(model, attack_results_DeepWordBug, tokenizer, max_length, use_purification=False)

# Evaluate VAT + Purification
purified_metrics = evaluate_adversarial_with_counts(vat_model, attack_results_DeepWordBug, tokenizer, max_length, use_purification=True)

# Calculate ASR metrics
asr_original = 1 - orig_metrics["accuracy"]
asr_defended = 1 - defense_metrics["accuracy"]
asr_purified = 1 - purified_metrics["accuracy"]
asr_reduction_vat = asr_original - asr_defended
asr_reduction_purified = asr_original - asr_purified

print(f"""
+---------------------------+-----------+-----------+-----------+
| Metric                    | Original  | VAT Only  | VAT+Purif |
+---------------------------+-----------+-----------+-----------+
| Accuracy                  | {orig_metrics["accuracy"]:.2%}    | {defense_metrics["accuracy"]:.2%}    | {purified_metrics["accuracy"]:.2%}    |
| ASR                       | {asr_original:.2%}    | {asr_defended:.2%}    | {asr_purified:.2%}    |
| ASR Reduction             | -         | {asr_reduction_vat:.2%}    | {asr_reduction_purified:.2%}    |
| Successful Defenses       | -         | {defense_metrics["defense_successful"]}       | {purified_metrics["defense_successful"]}       |
| Failed Defenses           | -         | {defense_metrics["defense_failed"]}       | {purified_metrics["defense_failed"]}       |
| Total Attempted           | {orig_metrics["total_attempted"]}      | {defense_metrics["total_attempted"]}      | {purified_metrics["total_attempted"]}      |
+---------------------------+-----------+-----------+-----------+
""")



=== Adversarial Evaluation ===

+---------------------------+-----------+-----------+-----------+
| Metric                    | Original  | VAT Only  | VAT+Purif |
+---------------------------+-----------+-----------+-----------+
| Accuracy                  | 7.42%    | 43.68%    | 84.33%    |
| ASR                       | 92.58%    | 56.32%    | 15.67%    |
| ASR Reduction             | -         | 36.26%    | 76.91%    |
| Successful Defenses       | -         | 418       | 807       |
| Failed Defenses           | -         | 539       | 150       |
| Total Attempted           | 957      | 957      | 957      |
+---------------------------+-----------+-----------+-----------+



In [ ]:
from collections import defaultdict

def analyze_character_frequency_in_purification(tokenizer, vocab_size=50000):
    """
    Analyze character frequency in the purification defense vocabulary
    to determine coverage statistics for presentation
    """

    # Get vocabulary sample (same as in your purification code)
    if hasattr(tokenizer, 'word_index'):
        vocab_sample = list(tokenizer.word_index.keys())[:vocab_size]
    else:
        vocab_sample = list(tokenizer.word_index.keys())[:vocab_size]

    # Count character frequencies
    char_freq = defaultdict(int)
    total_chars = 0

    for word in vocab_sample:
        for char in word.lower():
            if char.isalpha():
                char_freq[char] += 1
                total_chars += 1

    # Calculate percentage for each character
    char_percentage = {char: (count / total_chars) * 100 for char, count in char_freq.items()}

    # Sort characters by frequency (descending)
    sorted_chars = sorted(char_percentage.items(), key=lambda x: x[1], reverse=True)

    # Calculate cumulative coverage
    cumulative = 0
    coverage_list = []
    for char, perc in sorted_chars:
        cumulative += perc
        coverage_list.append((char, perc, cumulative))

    return coverage_list, total_chars, len(vocab_sample)

# Use this function with your tokenizer
def print_character_analysis(tokenizer):
    """Print detailed character frequency analysis for presentation"""

    coverage_list, total_chars, vocab_words = analyze_character_frequency_in_purification(tokenizer)

    print("=" * 60)
    print("CHARACTER FREQUENCY ANALYSIS FOR PURIFICATION DEFENSE")
    print("=" * 60)
    print(f"Vocabulary size analyzed: {vocab_words:,} words")
    print(f"Total alphabetic characters: {total_chars:,}")
    print("\nCharacter Distribution:")
    print("-" * 60)

    for i, (char, perc, cum) in enumerate(coverage_list):
        print(f"Rank {i+1:2d}: '{char}' - {perc:5.2f}% | Cumulative: {cum:5.2f}%")

    print("\n" + "=" * 60)
    print("COVERAGE ANALYSIS FOR PRESENTATION:")
    print("=" * 60)

    # Key statistics for presentation
    top_5_coverage = coverage_list[4][2]  # 5th character cumulative
    top_10_coverage = coverage_list[9][2]  # 10th character cumulative
    top_15_coverage = coverage_list[14][2] if len(coverage_list) >= 15 else coverage_list[-1][2]

    print(f"📊 Top 5 letters cover:  {top_5_coverage:.1f}% of all characters")
    print(f"📊 Top 10 letters cover: {top_10_coverage:.1f}% of all characters")
    print(f"📊 Top 15 letters cover: {top_15_coverage:.1f}% of all characters")

    # Most frequent characters for presentation
    top_5_chars = [char for char, _, _ in coverage_list[:5]]
    top_10_chars = [char for char, _, _ in coverage_list[:10]]

    print(f"\n🔤 Most frequent 5 letters: {', '.join(top_5_chars)}")
    print(f"🔤 Most frequent 10 letters: {', '.join(top_10_chars)}")

    print("\n" + "=" * 60)
    print("PRESENTATION TALKING POINTS:")
    print("=" * 60)
    print(f"• Character purification prioritizes the top {len(coverage_list)} letters")
    print(f"• Just the top 5 letters ('{', '.join(top_5_chars)}') cover {top_5_coverage:.1f}% of characters")
    print(f"• The top 10 letters cover {top_10_coverage:.1f}% of all character occurrences")
    print(f"• This frequency-based approach ensures efficient correction of common perturbations")

    return coverage_list

# Run the analysis (add this to your complete.py after loading the tokenizer)
if __name__ == "__main__":
    # Assuming you have your tokenizer loaded
    # tokenizer = your_loaded_tokenizer
    coverage_results = print_character_analysis(tokenizer)


CHARACTER FREQUENCY ANALYSIS FOR PURIFICATION DEFENSE
Vocabulary size analyzed: 50,000 words
Total alphabetic characters: 339,222

Character Distribution:
------------------------------------------------------------
Rank  1: 'e' - 11.36% | Cumulative: 11.36%
Rank  2: 'a' -  8.17% | Cumulative: 19.54%
Rank  3: 's' -  8.06% | Cumulative: 27.59%
Rank  4: 'i' -  7.71% | Cumulative: 35.31%
Rank  5: 'n' -  7.65% | Cumulative: 42.95%
Rank  6: 'r' -  6.96% | Cumulative: 49.91%
Rank  7: 't' -  6.34% | Cumulative: 56.25%
Rank  8: 'o' -  6.22% | Cumulative: 62.48%
Rank  9: 'l' -  5.02% | Cumulative: 67.50%
Rank 10: 'c' -  4.05% | Cumulative: 71.55%
Rank 11: 'd' -  3.90% | Cumulative: 75.44%
Rank 12: 'u' -  3.44% | Cumulative: 78.88%
Rank 13: 'm' -  2.84% | Cumulative: 81.72%
Rank 14: 'g' -  2.84% | Cumulative: 84.56%
Rank 15: 'p' -  2.84% | Cumulative: 87.39%
Rank 16: 'h' -  2.64% | Cumulative: 90.04%
Rank 17: 'b' -  2.06% | Cumulative: 92.10%
Rank 18: 'y' -  1.73% | Cumulative: 93.83%
Rank 19: '